# Data Leakage Examples

In [1]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

data_path = Path('/content/drive/MyDrive/courses/ml_basics/datasets/Ames housing/')
dataset_path = data_path/'AmesHousing.csv'
df = pd.read_csv(dataset_path)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from math import sqrt


# Example 1: Train-Test Contamination (Leaking future information during normalization)

In [4]:
# Splitting dataset into training and test sets
df = pd.read_csv(dataset_path)

# Incorrect: Using entire dataset to fit the scaler (including test data)
scaler = StandardScaler()
scaler.fit(df[['Lot Frontage', 'SalePrice']])  # Leaking information from test set

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
# Correct: Fit scaler only on training data
scaler.fit(train_df[['Lot Frontage', 'SalePrice']])

# Normalize features using only training data
X_train_scaled = scaler.transform(train_df[['Lot Frontage', 'SalePrice']])
X_test_scaled = scaler.transform(test_df[['Lot Frontage', 'SalePrice']])

# Example 2: Target Leakage (Feature containing information about the target)

In [5]:
# Including 'SalePrice' in the feature set, which is also the target variable
df = pd.read_csv(dataset_path)
X = df[['Lot Frontage', 'SalePrice', 'MS SubClass']]  # Incorrect, 'SalePrice' is the target
y = df['SalePrice']

# Correct: Remove target variable from features
X_correct = df[['Lot Frontage', 'MS SubClass']]

# Example 3: Time-Series Leakage (Using future information)

## Assume we have a time-series dataset where 'YearBuilt' could influence the prices in future years
## Incorrect: Including 'YearBuilt' for predicting future housing prices without adjusting for the timeline

In [6]:
# Adding an example with a hypothetical time-series split
df = pd.read_csv(dataset_path)
df['SaleYear'] = df['Yr Sold']  # Assume 'YrSold' represents the year of sale

# Incorrect: Using future information for prediction
# Here, we are using 'SaleYear' and 'YearBuilt' without respecting the temporal order
future_df = df.sort_values(by='SaleYear')
X_future = future_df[['Year Built', 'Lot Frontage', 'MS SubClass']]  # Incorrect, contains future information
y_future = future_df['SalePrice']

# Splitting without respecting time-series nature
train_df_future = future_df.iloc[:int(0.8 * len(future_df))]
test_df_future = future_df.iloc[int(0.8 * len(future_df)):]  # Data from the future is leaking into training

# Features and target
X_train_time = train_df_future[['Year Built', 'Lot Frontage', 'MS SubClass']]
y_train_time = train_df_future['SalePrice']
X_test_time = test_df_future[['Year Built', 'Lot Frontage', 'MS SubClass']]
y_test_time = test_df_future['SalePrice']

# Train model without time-series leakage
model_time = RandomForestRegressor(random_state=42)
model_time.fit(X_train_time, y_train_time)

# Predicting and evaluating
predictions_time = model_time.predict(X_test_time)
mae_time = mean_absolute_error(y_test_time, predictions_time)
mape_time = mean_absolute_percentage_error(y_test_time, predictions_time)
rmse_time = sqrt(mean_squared_error(y_test_time, predictions_time))
print(f"Mean Absolute Error (with Time-Series Leakage): {mae_time}")
print(f"Mean Absolute Percentage Error (with Time-Series Leakage): {mape_time}")
print(f"Root Mean Squared Error (with Time-Series Leakage): {rmse_time}")

Mean Absolute Error (without Time-Series Leakage): 39488.10633649679
Mean Absolute Percentage Error (without Time-Series Leakage): 0.22011038060846733
Root Mean Squared Error (without Time-Series Leakage): 58855.873552059


In [7]:
# Correct: Ensure training data is prior to test data
train_df_time = future_df[future_df['SaleYear'] < 2010]  # Assume data before 2010 is used for training
test_df_time = future_df[future_df['SaleYear'] >= 2010]

# Features and target
X_train_time = train_df_time[['Year Built', 'Lot Frontage', 'MS SubClass']]
y_train_time = train_df_time['SalePrice']
X_test_time = test_df_time[['Year Built', 'Lot Frontage', 'MS SubClass']]
y_test_time = test_df_time['SalePrice']

# Train model without time-series leakage
model_time = RandomForestRegressor(random_state=42)
model_time.fit(X_train_time, y_train_time)

# Predicting and evaluating
predictions_time = model_time.predict(X_test_time)
mae_time = mean_absolute_error(y_test_time, predictions_time)
mape_time = mean_absolute_percentage_error(y_test_time, predictions_time)
rmse_time = sqrt(mean_squared_error(y_test_time, predictions_time))
print(f"Mean Absolute Error (without Time-Series Leakage): {mae_time}")
print(f"Mean Absolute Percentage Error (without Time-Series Leakage): {mape_time}")
print(f"Root Mean Squared Error (without Time-Series Leakage): {rmse_time}")


Mean Absolute Error (without Time-Series Leakage): 36395.4213377162
Mean Absolute Percentage Error (without Time-Series Leakage): 0.23670886678934383
Root Mean Squared Error (without Time-Series Leakage): 54860.81798718368


# Example 4: Leakage from Derived Features (Improper feature engineering)

## Creating a feature that directly uses target information

## Incorrect: Creating a new feature that is derived from the target variable


In [8]:
train_df['Price_per_SqFt'] = train_df['SalePrice'] / train_df['Gr Liv Area']  # Leakage, as 'SalePrice' is the target

## Correct: Ensure that derived features do not use target variable information

In [9]:
# Creating a feature based on non-target columns
train_df['LotFrontage_per_MSSubClass'] = train_df['Lot Frontage'] / (train_df['MS SubClass'] + 1)

# Example 5: Data Leakage from Data Imputation

## Incorrect: Imputing missing values using the entire dataset (including test set)


In [10]:
imputer = SimpleImputer(strategy='mean')
X_transformed = imputer.fit(df[['Lot Frontage']])  # Leaking test set information during imputation

## Correct: Fit imputer only on training data

In [11]:
# Correct: Fit imputer only on training data
imputer.fit(train_df[['Lot Frontage']])

# Model Training and Evaluation (with Corrected Dataset)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Feature and target separation
X_train = train_df[['Lot Frontage', 'MS SubClass']]
y_train = train_df['SalePrice']
X_test = test_df[['Lot Frontage', 'MS SubClass']]
y_test = test_df['SalePrice']

# Create an imputer for missing values, fitting it only on the training data
imputer = SimpleImputer(strategy='mean')

# Fit imputer on the training data and transform both training and test data
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Normalizing features using only training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Training the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predicting and evaluating
predictions = model.predict(X_test_scaled)
mse = mean_squared_error(y_test, predictions)
mape = mean_absolute_percentage_error(y_test, predictions)
rmse = sqrt(mse)
print(f"Mean Squared Error: {mse}")
print(f"Mean Absolute Percentage Error: {mape}")
print(f"Root Mean Squared Error: {rmse}")


Mean Squared Error: 6786583038.496381
Mean Absolute Percentage Error: 0.3374186983018207
Root Mean Squared Error: 82380.72006541568
